# Code mẫu — Phân cụm

K-Means tìm các tâm sao cho tổng bình phương khoảng cách từ mỗi điểm tới tâm của nó nhỏ nhất. DBSCAN thì nối những vùng có đủ nhiều điểm đứng gần nhau. Các ví dụ dưới đây theo dõi từng bước gán nhóm, xem các trường hợp ở rìa, và cho thấy cách đo khoảng cách quyết định kết quả ra sao.

## Ví dụ 1: Một vòng K-Means với khởi tạo lệch

Năm điểm trên trục số là `[1, 2, 9, 10, 11]`. Hai tâm được khởi tạo ở 1 và 2, cả hai cùng nằm về một phía. Ta lặp ba vòng. Mỗi vòng gán từng điểm vào tâm gần nhất, rồi dời tâm về trung bình của các điểm thuộc nó. Inertia là tổng bình phương khoảng cách tới tâm, được in trước và sau khi dời tâm.

In [ ]:
import numpy as np

X = np.array([[1.], [2.], [9.], [10.], [11.]])
centers = np.array([[1.], [2.]])
for step in range(3):
    # Ma trận n×k: hàng i là khoảng cách bình phương từ điểm i tới từng tâm.
    distance2 = np.sum((X[:, None, :] - centers[None, :, :])**2, axis=2)
    labels = distance2.argmin(axis=1)
    before = float(distance2[np.arange(len(X)), labels].sum())
    new_centers = centers.copy()
    for cluster in range(len(centers)):
        members = X[labels == cluster]
        if len(members):
            new_centers[cluster] = members.mean(axis=0)
        # Cụm rỗng thì giữ tâm cũ, để khỏi lấy trung bình của mảng rỗng ra NaN.
    after = float(np.sum((X - new_centers[labels])**2))
    print("vòng", step + 1, "nhãn:", labels.tolist(),
          "tâm mới:", new_centers.ravel().round(3).tolist(),
          "inertia:", round(before, 3), "->", round(after, 3))
    assert after <= before + 1e-12
    centers = new_centers

vòng 1 nhãn: [0, 1, 1, 1, 1] tâm mới: [1.0, 8.0] inertia: 194.0 -> 50.0
vòng 2 nhãn: [0, 0, 1, 1, 1] tâm mới: [1.5, 10.0] inertia: 15.0 -> 2.5
vòng 3 nhãn: [0, 0, 1, 1, 1] tâm mới: [1.5, 10.0] inertia: 2.5 -> 2.5


Ở vòng 1, bốn điểm 2, 9, 10, 11 cùng gần tâm thứ hai hơn, nên tâm ấy dời về trung bình của chúng là 8. Sang vòng 2, điểm 2 giờ gần tâm 1 hơn tâm 8, nên nó đổi nhóm. Hai tâm dời về 1.5 và 10, inertia còn 2.5. Vòng 3 không còn điểm nào đổi nhóm, nên thuật toán dừng.

Mỗi bước đều làm inertia không tăng, dòng assert kiểm đúng điều đó. Nhưng điều này không bảo đảm K-Means luôn tìm được cách chia tốt nhất. Một khởi tạo tệ có thể làm nó mắc kẹt. Chẳng hạn, nếu hai tâm khởi tạo trùng nhau, một cụm có thể rỗng mãi. Thư viện xử lý chuyện này bằng cách khởi tạo khôn hơn và chạy nhiều lần, tốt hơn bản minh họa ở đây.

## Ví dụ 2: Một điểm ít hàng xóm vẫn có thể thuộc cụm

Năm điểm `[0, 0.1, 0.2, 0.35, 3]` được đưa vào DBSCAN với `min_samples = 3`. Lưu ý là khi đếm hàng xóm, chính điểm đang xét cũng được tính. Ta in số hàng xóm của từng điểm, và phân loại mỗi điểm thành core, border hay noise. Sau đó giảm `eps` để xem vùng mật độ thay đổi thế nào.

In [ ]:
import numpy as np
from sklearn.cluster import DBSCAN

X = np.array([[0.], [0.1], [0.2], [0.35], [3.]])
for eps in [0.16, 0.08]:
    distance = np.abs(X - X.T)
    counts = (distance <= eps).sum(axis=1)
    model = DBSCAN(eps=eps, min_samples=3).fit(X)
    core = np.zeros(len(X), dtype=bool)
    core[model.core_sample_indices_] = True
    kinds = ["core" if core[i] else "noise" if model.labels_[i] == -1 else "border"
             for i in range(len(X))]
    print("eps =", eps, "số hàng xóm:", counts.tolist())
    print("nhãn:", model.labels_.tolist(), "loại:", kinds)
    assert np.array_equal(core, counts >= 3)

eps = 0.16 số hàng xóm: [2, 3, 3, 2, 1]
nhãn: [0, 0, 0, 0, -1] loại: ['border', 'core', 'core', 'border', 'noise']
eps = 0.08 số hàng xóm: [1, 1, 1, 1, 1]
nhãn: [-1, -1, -1, -1, -1] loại: ['noise', 'noise', 'noise', 'noise', 'noise']


Với `eps = 0.16`, hai điểm 0.1 và 0.2 có đủ ba hàng xóm, nên là core. Điểm 0 và điểm 0.35 chỉ có hai hàng xóm, chưa đủ để làm core. Nhưng mỗi điểm nằm trong tầm của một core, nên chúng được kéo vào cụm dưới dạng border. Điểm 3 đứng một mình, nên là noise.

Giảm `eps` xuống 0.08 thì mỗi điểm chỉ còn đếm được chính nó, và cả năm điểm đều thành noise. Nhãn −1 nghĩa là noise dưới cấu hình đang dùng. Nó không phải mã lỗi, và cũng không phải một lớp có thật trong dữ liệu.

## Ví dụ 3: Đổi đơn vị làm đổi cách chia nhóm

Bốn cửa hàng được mô tả bằng hai số: số đơn mỗi giờ và diện tích tính bằng m². Tọa độ là `[0, 0]`, `[0, 1]`, `[10, 0]`, `[10, 1]`. Đây là dữ liệu dựng ra để kiểm khoảng cách, không phải cửa hàng thật. Ta chạy K-Means trên số liệu thô và trên số liệu đã chuẩn hóa, rồi làm lại sau khi đổi diện tích sang cm².

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import StandardScaler

X = np.array([[0., 0.], [0., 1.], [10., 0.], [10., 1.]])
raw_partitions, scaled_partitions = [], []
for unit, factor in [("m²", 1.), ("cm²", 10000.)]:
    measured = X * np.array([1., factor])
    raw = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(measured)
    scaled = StandardScaler().fit_transform(measured)
    normalized = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(scaled)
    raw_partitions.append(raw)
    scaled_partitions.append(normalized)
    print(unit, "nhãn thô / chuẩn hóa:", raw.tolist(), normalized.tolist())
print("ARI giữa hai thang thô:", round(adjusted_rand_score(*raw_partitions), 3))
print("ARI giữa hai thang chuẩn hóa:", round(adjusted_rand_score(*scaled_partitions), 3))
assert adjusted_rand_score(*scaled_partitions) == 1.

m² nhãn thô / chuẩn hóa: [1, 1, 0, 0] [0, 0, 1, 1]
cm² nhãn thô / chuẩn hóa: [0, 1, 0, 1] [0, 0, 1, 1]
ARI giữa hai thang thô: -0.5
ARI giữa hai thang chuẩn hóa: 1.0


Tính bằng m², số đơn chênh tới 10 còn diện tích chỉ chênh 1, nên K-Means chia theo số đơn. Tính bằng cm², diện tích chênh tới 10000, nên K-Means quay sang chia theo diện tích. Hai cách chia cắt chéo nhau, và ARI giữa chúng là −0.5. ARI đo mức hai cách chia khớp nhau: 1 là khớp hoàn toàn, quanh 0 là như ngẫu nhiên.

Sau khi chuẩn hóa, cả hai đơn vị cho cùng một cách chia, và ARI bằng 1. Lưu ý là bốn điểm chuẩn hóa tạo thành một hình vuông. Chia theo chiều ngang hay chiều dọc đều cho cùng inertia. Seed cố định chỉ giúp lần chạy nào cũng chọn giống nhau, chứ không có nghĩa đó là cách chia “đúng”.

## Ví dụ 4: Hai dải cong và hai cách chấm điểm

Ta sinh 240 điểm thành hai dải hình trăng khuyết, rồi so K-Means với DBSCAN. Nhãn gốc lúc sinh dữ liệu chỉ dùng để tính ARI, không đưa vào lúc fit. DBSCAN có thể đánh dấu một số điểm là noise, nên ta báo số điểm noise trước khi đọc silhouette.

In [ ]:
import numpy as np
from sklearn.cluster import DBSCAN, KMeans
from sklearn.datasets import make_moons
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

X, reference = make_moons(n_samples=240, noise=0.045, random_state=30)
Z = StandardScaler().fit_transform(X)
models = [
    ("K-Means", KMeans(n_clusters=2, n_init=10, random_state=0)),
    ("DBSCAN", DBSCAN(eps=0.28, min_samples=5)),
]
for name, model in models:
    labels = model.fit_predict(Z)
    retained = labels != -1
    clusters = len(set(labels[retained]))
    print(name, "số cụm / số noise:", clusters, int((~retained).sum()),
          "ARI toàn bộ:", round(adjusted_rand_score(reference, labels), 3))
    if 2 <= clusters < retained.sum():
        # Bỏ các điểm noise khi tính silhouette, và ghi rõ đã chấm trên tập nào.
        silhouette = silhouette_score(Z[retained], labels[retained])
        print("silhouette trên điểm không-noise:", round(silhouette, 3))
    else:
        print("silhouette: chưa đủ điều kiện")

K-Means số cụm / số noise: 2 0 ARI toàn bộ: 0.453
silhouette trên điểm không-noise: 0.489
DBSCAN số cụm / số noise: 2 1 ARI toàn bộ: 0.992
silhouette trên điểm không-noise: 0.383


DBSCAN gần như tách đúng hai dải, ARI 0.992, chỉ để lại một điểm noise. K-Means chỉ đạt ARI 0.453, vì nó luôn cắt dữ liệu bằng một đường thẳng giữa hai tâm.

Thế nhưng silhouette lại xếp K-Means cao hơn, 0.489 so với 0.383. Silhouette thưởng cho những cụm tròn, gọn, và cách xa nhau. Nó không biết gì về hình dạng cong của hai dải. Vì vậy, chọn tiêu chí nào để chấm cũng là một quyết định, và phải hợp với mục tiêu của bài. Còn một chi tiết nữa: silhouette của DBSCAN chỉ chấm 239 điểm không phải noise, còn ARI chấm đủ 240. Khi so hai con số, cần nhớ chúng được tính trên hai tập khác nhau.